# Logistic regression
Target variables in logistic regressions are binary.

Example: Titanic dataset. Model:

${Survived}_i^{(0,1)}=\beta_0+\beta_1{Fare}_i+\beta_2{Pclass}_i+\beta_3{Gender}_i$

The output is the probability of survival


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../06-Exploratory_data_analysis/data/Titanic-Dataset.csv")

# Select variables
df_clean = df[["Survived", "Fare", "Pclass", "Sex"]].dropna()

# one_hot_encoding
df_clean = pd.get_dummies(df_clean, columns=['Sex'], drop_first=True, dtype=int)

df_clean.head()

,Survived,Fare,Pclass,Sex_male
0,0,7.2500,3,1
1,1,71.2833,1,0
2,1,7.9250,3,0
3,1,53.1000,1,0
4,0,8.0500,3,1


Let's inspect their correlation:

It seems Fare is weak to moderately correlated with survival and the correlation is statistically significant.

In [2]:
from scipy.stats import pearsonr
from itertools import combinations

for var1, var2 in combinations(df_clean.columns, 2):
    corr, p_value = pearsonr(df_clean[var1], df_clean[var2])
    print(f"{var1} vs {var2}: {corr=:.3f}, {p_value=:.3f}")


Survived vs Fare: corr=0.257, p_value=0.000
Survived vs Pclass: corr=-0.338, p_value=0.000
Survived vs Sex_male: corr=-0.543, p_value=0.000
Fare vs Pclass: corr=-0.549, p_value=0.000
Fare vs Sex_male: corr=-0.182, p_value=0.000
Pclass vs Sex_male: corr=0.132, p_value=0.000


In [3]:
# run logistic regression
import statsmodels.api as sm

X = df_clean[["Fare", "Pclass", "Sex_male"]]
X = sm.add_constant(X)

y = df_clean["Survived"]

model = sm.Logit(y, X).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.463903
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               Survived   No. Observations:                  891
Model:                          Logit   Df Residuals:                      887
Method:                           MLE   Df Model:                            3
Date:                Wed, 08 Apr 2026   Pseudo R-squ.:                  0.3034
Time:                        10:41:51   Log-Likelihood:                -413.34
converged:                       True   LL-Null:                       -593.33
Covariance Type:            nonrobust   LLR p-value:                 1.029e-77
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.1437      0.364      8.645      0.000       2.431       3.856
Fare           0.0015      0.

In [4]:
# convert to odd ratio

odds_ratios = np.exp(model.params)
print(odds_ratios)

const       23.190233
Fare         1.001451
Pclass       0.399647
Sex_male     0.072032
dtype: float64


Interpretation:
- OR > 1 → increases odds
- OR < 1 → decreases odds

Example:
- OR = 2 → doubles odds, aka, increase the odds by 100%
- OR = 0.5 → halves odds

What is odds? Chance the event happens vs chance it does not happen: ${Odd} = \frac{P}{1-P}$. If the probability of event $A$ happen is 75%, then, the odd of $A$ happening is 3 (the odd is 3 to 1), meaning the event is 3 times more likely to happen than not.

Therefore, the interpretation of our model:
- Fare has no significant relationship with survival ($p=0.475$ so $p>0.05$).
- Higher class (lower class number) passengers have higher survival rate (statistically significant). Increase class number (e.g., first class to second class) will reduce the odds of survival by 60% ($1-0.399647$)
- Male has lower survival rate (statistically significant). Being a male reduce the survival rate by 92%.

Our model:
- Model converged → estimation worked properly
- Pseudo $R^2$ = 0.303: The model explains a moderate amount of variation in survival
- LLR p-value ≈ 0 (1.029e-77) → The model as a whole is highly significant
- At least one predictor is strongly related to survival




In [ ]:
df_clean["pred_prob"] = model.predict(X)

print(df_clean[["Survived", "pred_prob"]].head())

In [ ]:
sns.set_style("whitegrid")
sns.boxenplot(data=df_clean, x="Survived", y="pred_prob")

We can do a classification algorithm using logistic regression

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

df_clean["pred_class"] = (df_clean["pred_prob"] >= 0.5).astype(int)

accuracy = accuracy_score(df_clean["Survived"], df_clean["pred_class"])

print("Accuracy:", accuracy)

# Detailed report
print(classification_report(df_clean["Survived"], df_clean["pred_class"]))

# Poisson regression

Research Question: How do population density, median age, and vaccination rate affect COVID case counts?

Model:

$log(\lambda_i)=\beta_0+\beta_1 \cdot {pop\_density}_i+\beta_2\cdot{median\_age}_i+\beta_3 \cdot {vac\_per\_hund}_i + \epsilon$

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

cols = ["total_deaths", "pop_density", "median_age", "gdp_per_capita"]

df=pd.read_csv("../05-data_cleaning/data/covidtotals.csv")

reg_df = df[cols].copy().dropna()

reg_df['total_deaths'] = reg_df['total_deaths'].astype(int)
reg_df

In [ ]:
import statsmodels.api as sm

X = reg_df[["pop_density", "median_age", "gdp_per_capita"]]
X = sm.add_constant(X)

y = reg_df["total_deaths"]

model = sm.GLM(y, X, family=sm.families.Poisson()).fit()

print(model.summary())

In [ ]:
# Convert to Incident Rate Ratio (IRR)

irr = np.exp(model.params)
print(irr)

- IRR = 1.10: 10% increase in expected cases
- IRR = 0.80: 20% decrease

Add Offset: Since countries have different population sizes, we wanted to add population to the equation.

In [ ]:
import numpy as np

df["log_population"] = np.log(df["population"])

reg_df = df[cols+['log_population']].copy().dropna()

X = reg_df[["pop_density", "median_age", "gdp_per_capita"]]
X = sm.add_constant(X)

y = reg_df["total_deaths"]

model = sm.GLM(
    y,
    X,
    family=sm.families.Poisson(),
    offset=reg_df["log_population"]
).fit()

print(model.summary())

In [ ]:
# Convert to Incident Rate Ratio (IRR)

irr = np.exp(model.params)
print(irr)

In [ ]:
reg_df["pred_deaths"] = model.predict(X)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(reg_df["pop_density"], reg_df["total_deaths"], alpha=0.3, label='Actual deaths')
plt.scatter(reg_df["pop_density"], reg_df["pred_deaths"], alpha=0.3, label='Predicted deaths')

plt.xlabel("Population Density")
plt.ylabel("Deaths")
plt.title("Observed vs Predicted Deaths")

plt.xscale("log")
plt.yscale("log")

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 8), sharex=False, sharey='all')
sns.set_style("whitegrid")

for ax, iv in zip(axes.flat, ["pop_density", "median_age", "gdp_per_capita","log_population"]):
    # plot_df = reg_df[iv, ]
    g0=sns.scatterplot(x=reg_df[iv], y=reg_df['total_deaths'], label='Actual deaths', ax=ax, alpha=0.35, color='tab:blue')
    g1=sns.scatterplot(x=reg_df[iv], y=reg_df['pred_deaths'], label='Predicted deaths', ax=ax, alpha=0.35, color="tab:orange")
    ax.set_yscale('log')
    if iv=='pop_density':
        ax.set_xscale("log")

# Negative Binomial regression

In the case of COVID deaths, the target variable (total deaths) mean is way much smaller than the variance. In cases like this, Negative Binomial regression is more ideal than Poisson regression

In [ ]:
print(f"Mean: {reg_df['total_deaths'].mean()}")
print(f"Variance: {reg_df['total_deaths'].var()}")
sns.histplot(data=reg_df['total_deaths'])

In [ ]:
model_nb = sm.GLM(
    y,
    X,
    family=sm.families.NegativeBinomial(alpha=0.5),
).fit()

print(model_nb.summary())

In [ ]:
# Convert to Incident Rate Ratio (IRR)

irr = np.exp(model_nb.params)
print(irr)

In [ ]:
reg_df["pred_deaths"] = model_nb.predict(X)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 8), sharex=False, sharey='all')
sns.set_style("whitegrid")

for ax, iv in zip(axes.flat, ["pop_density", "median_age", "gdp_per_capita","log_population"]):
    # plot_df = reg_df[iv, ]
    g0=sns.scatterplot(x=reg_df[iv], y=reg_df['total_deaths'], label='Actual deaths', ax=ax, alpha=0.35, color='tab:blue')
    g1=sns.scatterplot(x=reg_df[iv], y=reg_df['pred_deaths'], label='Predicted deaths', ax=ax, alpha=0.35, color="tab:orange")
    ax.set_yscale('log')
    if iv=='pop_density':
        ax.set_xscale("log")


In [ ]:
fig, axes=plt.subplots(1, 2, figsize=(8.5, 3), sharex=False, sharey=False)
sns.kdeplot(data=model.resid_deviance, ax=axes.flat[0],fill=True, alpha=0.4, label='Poisson')
sns.kdeplot(data=model_nb.resid_deviance, ax=axes.flat[1], fill=True, alpha=0.4, label='Negative Binomial')

axes.flat[0].set_xlabel("Deviance Residuals (Poisson)")
axes.flat[1].set_xlabel("Deviance Residuals (Negative Binomial)")

In [ ]:
fig, axes=plt.subplots(2, 2, figsize=(12, 10))

g0=sns.histplot(data=model.resid_deviance, bins=50, ax=axes.flat[0])
g0.text(x=-1_500, y=30, s="Poisson: Deviance Residue", fontsize=15)
g1=sm.qqplot(model.resid_deviance, line='q', ax=axes.flat[2])

g2=sns.histplot(data=model_nb.resid_deviance, bins=50,ax=axes.flat[1])
g2.text(x=0, y=15, s="BN: Deviance Residue", fontsize=15)
g3=sm.qqplot(model_nb.resid_deviance, line='q', ax=axes.flat[3])